# Machine Learning Para Predição de Desempenho de Alunos do 3º/4º do Ensino Médio

## Integrantes
- Ana Caroline Souza Lira
- Daniel Goulart Camacho Gonçalves
- Fernando Giongo
- Theo Luigi
- João Gabriel Lourenço Marques

---
O projeto tem por objetivo prever os níveis Saeb de proficiência em Lingua Portuguesa (LP) ou Matemática (MT) de alunos do 3º e 4º Anos do Ensino Médio. Para isso, utilizaremos classificadores de Aprendizado de Máquina em cima dos microdados do Saeb de 2023 e a biblioteca Pandas do Python.

# Pré-processamento de dados

In [18]:
# Importando as bibliotecas necessárias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_aluno_original = pd.read_csv('dados/TS_ALUNO_34EM.csv', encoding='latin-1', sep=';')
df_diretor_original = pd.read_csv('dados/TS_DIRETOR.csv', encoding='latin-1', sep=';')
df_escola_original = pd.read_csv('dados/TS_ESCOLA.csv', encoding='latin-1', sep=';')

O próximo passo será definir quais features serão mantidas para análise. Para preservar o escopo do estudo, investigamos o dicionário de dados em busca de variáveis que representassem o contexto socioeconômico familiar, a infraestrutura física das escolas e a capacitação dos professores. Esses fatores foram selecionados primariamente por serem aspectos que, em tese, apresentam grande potencial de influência sobre o desempenho dos alunos. Ao longo do projeto, avaliaremos o comportamento dessas variáveis e sua efetiva relevância com os modelos desenvolvidos.

Nesse momento, também definimos critérios de limpeza dos dados. No conjunto de alunos, foram removidos os registros de estudantes que não compareceram às provas ou que responderam pelo menos três questões do questionário socioeconômico. De forma semelhante, no conjunto de diretores, excluímos os registros daqueles que não preencheram o questionário ou que não eram responsáveis por alunos do 3º ou o 4º ano do Ensino Médio. Por fim, no conjunto de escolas, foram removidas as instituições que não possuíam alunos matriculados nessas séries.

In [ ]:
# ================================================================
# Escolhemos a features que temos interesses e as features
# que usaremos para limpar linhas que não tenham dados relevantes
# ================================================================

colunas_de_interesse_aluno = ['ID_ESCOLA', 'ID_ALUNO', 'ID_UF', 'ID_MUNICIPIO', 'ID_AREA', 'IN_PUBLICA', 'ID_SERIE', 'PROFICIENCIA_LP_SAEB', 
'PROFICIENCIA_MT_SAEB', 'TX_RESP_Q01', 'TX_RESP_Q02', 'TX_RESP_Q03', 'TX_RESP_Q04', 'TX_RESP_Q05a', 'TX_RESP_Q05b',
'TX_RESP_Q05c', 'TX_RESP_Q06','TX_RESP_Q07a','TX_RESP_Q07b','TX_RESP_Q07c','TX_RESP_Q07d','TX_RESP_Q07e','TX_RESP_Q08','TX_RESP_Q09',
'TX_RESP_Q10a','TX_RESP_Q10b','TX_RESP_Q10c','TX_RESP_Q10d','TX_RESP_Q10e','TX_RESP_Q10f','TX_RESP_Q11a','TX_RESP_Q11b','TX_RESP_Q11c',
'TX_RESP_Q12b','TX_RESP_Q12c','TX_RESP_Q14','TX_RESP_Q15b','TX_RESP_Q16','TX_RESP_Q18','TX_RESP_Q21a',
'TX_RESP_Q21b','TX_RESP_Q21c','TX_RESP_Q21d','TX_RESP_Q21e','TX_RESP_Q23d','TX_RESP_Q24']

remover_aluno = {
    'IN_PRESENCA_LP': 0,                    # Remover alunos que não tenham respondido a prova de língua portuguesa
    'IN_PRESENCA_MT': 0,                    # Remover alunos que não tenham respondido a prova de matemática
    'IN_PROFICIENCIA_LP': 0,                # Remover alunos que não tenham proficiência em língua portuguesa
    'IN_PROFICIENCIA_MT': 0,                # Remover alunos que não tenham proficiência em matemática
    'IN_PREENCHIMENTO_QUESTIONARIO': 0}     # Remover alunos que não preencheram o questionário

colunas_de_interesse_diretor = ['ID_ESCOLA', 'ID_SERIE', 'TX_Q020','TX_Q022','TX_Q032','TX_Q033','TX_Q035',
'TX_Q036','TX_Q056','TX_Q057','TX_Q078','TX_Q079','TX_Q081','TX_Q082','TX_Q083', 'TX_Q085','TX_Q087',
'TX_Q108','TX_Q119','TX_Q129','TX_Q130','TX_Q139','TX_Q191','TX_Q194','TX_Q203','TX_Q205','TX_Q206','TX_Q207',
'TX_Q208','TX_Q209']

remover_diretor = {
    'IN_PREENCHIMENTO_QUESTIONARIO': [0],   # Remover diretores que não preencheram o questionário
    'ID_SERIE': [5, 9, 2]}

colunas_de_interesse_escola = ['ID_ESCOLA','PC_FORMACAO_DOCENTE_MEDIO']

remover_escola = {
    'NU_PRESENTES_EM': 0}                   # Remover escolas que não tenham alunos presentes no ensino médio


# ================================================================
# Aplicaremos as regras de limpeza de dados e selecionaremos as features
# ================================================================

print("\n# ================================================================\n")
df_aluno_limpo = df_aluno_original.copy()
df_diretor_limpo = df_diretor_original.copy()
df_escola_limpo = df_escola_original.copy()

for coluna, valor in remover_aluno.items():
    antes = len(df_aluno_limpo)
    df_aluno_limpo = df_aluno_limpo[df_aluno_limpo[coluna] != valor]
    removidas = antes - len(df_aluno_limpo)
    
    print(
        f"Alunos: removidas {removidas} linhas "
        f"pela regra: {coluna} != {valor}"
    )

for coluna, valor in remover_diretor.items():
    antes = len(df_diretor_limpo)
    df_diretor_limpo = df_diretor_limpo[~df_diretor_limpo[coluna].isin(valor)]
    removidas = antes - len(df_diretor_limpo)

    print(
        f"Diretores: removidas {removidas} linhas "
        f"pela regra: {valor} não estar em {coluna}"
    )

for coluna, valor in remover_escola.items():
    antes = len(df_escola_limpo)
    df_escola_limpo = df_escola_limpo[df_escola_limpo[coluna] != valor]
    removidas = antes - len(df_escola_limpo)

    print(
        f"Escolas: removidas {removidas} linhas "
        f"pela regra: {coluna} != {valor}"
    )

df_aluno_limpo = df_aluno_limpo[colunas_de_interesse_aluno]
df_diretor_limpo = df_diretor_limpo[colunas_de_interesse_diretor]
df_escola_limpo = df_escola_limpo[colunas_de_interesse_escola]

print("\n# ================================================================\n")
print(f'Alunos: {len(df_aluno_limpo)} linhas')
print(f'Diretores: {len(df_diretor_limpo)} linhas')
print(f'Escolas: {len(df_escola_limpo)} linhas')
print("\n# ================================================================\n")

# Quantidade de linhas removidas do dataset original
print(f'Alunos: {len(df_aluno_original) - len(df_aluno_limpo)} linhas removidas do dataset original -> {(len(df_aluno_limpo))/len(df_aluno_original)*100:.2f}% linhas restantes')
print(f'Diretores: {len(df_diretor_original) - len(df_diretor_limpo)} linhas removidas do dataset original -> {(len(df_diretor_limpo))/len(df_diretor_original)*100:.2f}% linhas restantes')
print(f'Escolas: {len(df_escola_original) - len(df_escola_limpo)} linhas removidas do dataset original -> {(len(df_escola_limpo))/len(df_escola_original)*100:.2f}% linhas restantes')

print("\n# ================================================================\n")


# ================================================================

Alunos: removidas 562414 linhas pela regra: IN_PRESENCA_LP != 0
Alunos: removidas 0 linhas pela regra: IN_PRESENCA_MT != 0
Alunos: removidas 2966 linhas pela regra: IN_PROFICIENCIA_LP != 0
Alunos: removidas 0 linhas pela regra: IN_PROFICIENCIA_MT != 0
Alunos: removidas 11509 linhas pela regra: IN_PREENCHIMENTO_QUESTIONARIO != 0
Diretores: removidas 17304 linhas pela regra: [0] não estar em IN_PREENCHIMENTO_QUESTIONARIO
Diretores: removidas 72208 linhas pela regra: [5, 9, 2] não estar em ID_SERIE
Escolas: removidas 122 linhas pela regra: NU_PRESENTES_EM != 0

# ================================================================

Alunos: 1514448 linhas
Diretores: 17577 linhas
Escolas: 70029 linhas

# ================================================================

Alunos: 576889 linhas removidas do dataset original -> 72.42% linhas restantes
Diretores: 89512 linhas removidas do dataset original -> 16.41% linhas restantes


In [20]:
# ================================================================
# Podemos ter problemas ao unificar as tabelas se houver escolas
# com o mesmo ID e multiplos diretores
# ================================================================

print("\n# ================================================================\n")
print(f"Alunos em mais de uma escola: {df_aluno_limpo['ID_ALUNO'].duplicated().sum()}")
print(f"Alunos duplicados: {df_aluno_limpo['ID_ALUNO'].duplicated().sum()}")


print(f"Escolas com ID_ESCOLA duplicados: {df_escola_limpo['ID_ESCOLA'].duplicated().sum()}")

escolas_com_diretores_duplicados = (
    df_diretor_limpo['ID_ESCOLA']
    .value_counts()
    .loc[lambda x: x > 1]
)

escolas_sem_diretores = (
    df_escola_limpo['ID_ESCOLA']
    .loc[~df_escola_limpo['ID_ESCOLA'].isin(df_diretor_limpo['ID_ESCOLA'])]
)

print(f'Escolas com mais de um diretor: {len(escolas_com_diretores_duplicados)}')
print("\n# ================================================================\n")


# ================================================================

Alunos em mais de uma escola: 0
Alunos duplicados: 0
Escolas com ID_ESCOLA duplicados: 0
Escolas com mais de um diretor: 723

# ================================================================



In [ ]:
# ================================================================
# Vamos tratar as Features transformando de categóricas para numéricas quando possivel
# ================================================================

colunas_diretor_numericas = [
    'TX_Q020',
    'TX_Q022'
]

colunas_diretor_ordinais_AD = [
    'TX_Q056',
    'TX_Q057',
    'TX_Q108'
]

colunas_diretor_ordinais_AE = [
    'TX_Q032',
    'TX_Q033',
    'TX_Q035',
    'TX_Q036',
    'TX_Q191'
]

colunas_diretor_booleanas = [
    'TX_Q078', 'TX_Q079', 'TX_Q081', 'TX_Q082',
    'TX_Q083', 'TX_Q085', 'TX_Q087', 'TX_Q119',
    'TX_Q129', 'TX_Q130', 'TX_Q139', 'TX_Q194',
    'TX_Q203', 'TX_Q205', 'TX_Q206', 'TX_Q207',
    'TX_Q208', 'TX_Q209'
]

colunas_aluno_booleanas = [
    'TX_RESP_Q07a', 'TX_RESP_Q07b', 'TX_RESP_Q07c',
    'TX_RESP_Q07d', 'TX_RESP_Q07e',
    'TX_RESP_Q11a', 'TX_RESP_Q11b', 'TX_RESP_Q11c',
    'TX_RESP_Q15b'
]

colunas_aluno_booleanas_ABC = [
]

colunas_aluno_ordinais_AD = [
    'TX_RESP_Q23d'
]

colunas_aluno_categoricas = [
    'TX_RESP_Q01', 'TX_RESP_Q02', 'TX_RESP_Q03',
    'TX_RESP_Q04', 'TX_RESP_Q05a', 'TX_RESP_Q05b',
    'TX_RESP_Q05c', 'TX_RESP_Q06', 'TX_RESP_Q08',
    'TX_RESP_Q09', 'TX_RESP_Q10a', 'TX_RESP_Q10b',
    'TX_RESP_Q10c', 'TX_RESP_Q10d', 'TX_RESP_Q10e',
    'TX_RESP_Q10f', 'TX_RESP_Q12b', 'TX_RESP_Q12c',
    'TX_RESP_Q14', 'TX_RESP_Q16',
    'TX_RESP_Q18', 'TX_RESP_Q21a', 'TX_RESP_Q21b',
    'TX_RESP_Q21c', 'TX_RESP_Q21d', 'TX_RESP_Q21e',
    'TX_RESP_Q24'
]
    

# ================================================================
# Mapeamentos normalizados [0,1]
# ================================================================

mapa_AD = {
    'A': 0 / 3,
    'B': 1 / 3,
    'C': 2 / 3,
    'D': 3 / 3
}

mapa_AE = {
    'A': 0 / 4,
    'B': 1 / 4,
    'C': 2 / 4,
    'D': 3 / 4,
    'E': 4 / 4
}

mapa_bool = {
    'A': 0.0,
    'B': 1.0
}

mapa_bool_ABC = {
    'A': 0.0,
    'B': 1.0,
    'C': 1.0
}

valores_invalidos = ['*', '.', 'F', '']

# ================================================================
# Conversão das respostas dos diretores
# ================================================================

for col in colunas_diretor_ordinais_AD:
    if col in df_diretor_limpo.columns:
        df_diretor_limpo[col] = (
            df_diretor_limpo[col]
            .map(mapa_AD)
            .astype('float32')
        )

for col in colunas_diretor_ordinais_AE:
    if col in df_diretor_limpo.columns:
        df_diretor_limpo[col] = (
            df_diretor_limpo[col]
            .map(mapa_AE)
            .astype('float32')
        )

for col in colunas_diretor_booleanas:
    if col in df_diretor_limpo.columns:
        df_diretor_limpo[col] = (
            df_diretor_limpo[col]
            .map(mapa_bool)
            .astype('float32')
        )

for col in colunas_aluno_categoricas:
    if col in df_aluno_limpo.columns:
        df_aluno_limpo[col] = (
            df_aluno_limpo[col]
            .replace(valores_invalidos, np.nan)
        )

for col in colunas_aluno_booleanas:
    if col in df_aluno_limpo.columns:
        df_aluno_limpo[col] = (
            df_aluno_limpo[col]
            .replace(valores_invalidos, np.nan)
            .map(mapa_bool)
            .astype('float32')
        )

for col in colunas_aluno_booleanas_ABC:
    if col in df_aluno_limpo.columns:
        df_aluno_limpo[col] = (
            df_aluno_limpo[col]
            .replace(valores_invalidos, np.nan)
            .map(mapa_bool_ABC)
            .astype('float32')
        )

for col in colunas_aluno_ordinais_AD:
    if col in df_aluno_limpo.columns:
        df_aluno_limpo[col] = (
            df_aluno_limpo[col]
            .replace(valores_invalidos, np.nan)
            .map(mapa_AD)
            .astype('float32')
        )

df_aluno_limpo['ID_AREA'] = (
    df_aluno_limpo['ID_AREA']
    .replace({
        1: 0,
        2: 1,
    })
    .astype('float32')
)

df_aluno_limpo['ID_UF'] = df_aluno_limpo['ID_UF'].astype('int8')
df_aluno_limpo['ID_MUNICIPIO'] = df_aluno_limpo['ID_MUNICIPIO'].astype('int32')

# Cada linha restante possui um diretor válido
df_diretor_limpo['POSSUI_DIRETOR'] = np.int8(1)

print("\n# ================================================================\n")
print('Registros duplicados de (ID_ESCOLA, ID_SERIE):',df_diretor_limpo.duplicated(subset=['ID_ESCOLA', 'ID_SERIE']).sum())

# ================================================================
# Estatísticas
# ================================================================

pares_aluno = (
    df_aluno_limpo[['ID_ESCOLA', 'ID_SERIE']]
    .drop_duplicates()
)

pares_diretor = (
    df_diretor_limpo[['ID_ESCOLA', 'ID_SERIE']]
)

pares_com_diretor = (
    pares_aluno.merge(
        pares_diretor,
        on=['ID_ESCOLA', 'ID_SERIE'],
        how='left',
        indicator=True
    )
)

qtd_com = (pares_com_diretor['_merge'] == 'both').sum()
qtd_sem = (pares_com_diretor['_merge'] == 'left_only').sum()

print(
    f'Pares escola-série com diretor: '
    f'{qtd_com:,} -> '
    f'{100*qtd_com/len(pares_com_diretor):.2f}%'
)

print(
    f'Pares escola-série sem diretor: '
    f'{qtd_sem:,} -> '
    f'{100*qtd_sem/len(pares_com_diretor):.2f}%'
)

# ================================================================
# Merge final
# ================================================================

df_diretor_limpo_copy = df_diretor_limpo.copy()
df_escola_limpo_copy = df_escola_limpo.copy()

df_unificado = (
    df_aluno_limpo
    .merge(
        df_diretor_limpo_copy,
        on=['ID_ESCOLA', 'ID_SERIE'],
        how='left'
    )
    .merge(
        df_escola_limpo_copy,
        on='ID_ESCOLA',
        how='left'
    )
)

# Escolas/séries sem diretor recebem 0
df_unificado['POSSUI_DIRETOR'] = (
    df_unificado['POSSUI_DIRETOR']
    .fillna(0)
    .astype(np.int8)
)

registros_com = int(df_unificado['POSSUI_DIRETOR'].sum())
registros_sem = len(df_unificado) - registros_com

pct_com = 100 * registros_com / len(df_unificado)
pct_sem = 100 - pct_com

print(
    f'Registros com diretor: '
    f'{registros_com:,} -> {pct_com:.2f}%'
)

print(
    f'Registros sem diretor: '
    f'{registros_sem:,} -> {pct_sem:.2f}%'
)

print(f'Linhas finais: {len(df_unificado):,}')
print("\n# ================================================================\n")


# ================================================================

Registros duplicados de (ID_ESCOLA, ID_SERIE): 0
Pares escola-série com diretor: 17,424 -> 82.82%
Pares escola-série sem diretor: 3,615 -> 17.18%
Registros com diretor: 1,248,091 -> 82.41%
Registros sem diretor: 266,357 -> 17.59%
Linhas finais: 1,514,448

# ================================================================



In [ ]:
# ================================================================
# Adicionando a classificação de nível de acordo com a escala SAEB
# ================================================================
df_final = df_unificado.copy()

df_final = df_final[df_final['PROFICIENCIA_LP_SAEB'] >= 225]
df_final = df_final[df_final['PROFICIENCIA_MT_SAEB'] >= 225]

def classificar_nivel_LP(nota_proficiencia):
    if nota_proficiencia < 300:
        return 1
    elif 300 <= nota_proficiencia < 350:
        return 2
    elif nota_proficiencia >= 350:
        return 3
    
def classificar_nivel_MT(nota_proficiencia):
    if nota_proficiencia < 300:
        return 1
    elif 300 <= nota_proficiencia < 375:
        return 2
    elif nota_proficiencia >= 375:
        return 3


   
# Adicionando os níveis de proficiência de cada aluno e removendo as colunas originais de proficiência
df_final['NIVEL_LP'] = df_final['PROFICIENCIA_LP_SAEB'].apply(classificar_nivel_LP)
df_final['NIVEL_MT'] = df_final['PROFICIENCIA_MT_SAEB'].apply(classificar_nivel_MT)
df_final = df_final.drop(columns=['PROFICIENCIA_LP_SAEB', 'PROFICIENCIA_MT_SAEB'])

df_final = df_final[(df_final['NIVEL_LP'] != 0) & (df_final['NIVEL_MT'] != 0)]

df_final.tail(20)

,ID_ESCOLA,ID_ALUNO,ID_UF,ID_MUNICIPIO,ID_AREA,IN_PUBLICA,ID_SERIE,TX_RESP_Q01,TX_RESP_Q02,TX_RESP_Q03,...,TX_Q206,TX_Q207,TX_Q208,TX_Q209,POSSUI_DIRETOR,PC_FORMACAO_DOCENTE_MEDIO,MEDIA_EM_NIVEL_LP,MEDIA_EM_NIVEL_MT,NIVEL_LP,NIVEL_MT
1514419,61470349,56425817,53,6327738,0.0,1,12,A,C,A,...,1.0,1.0,0.0,1.0,1,88.8,1.0,1.0,2,1
1514421,61470349,56425947,53,6327738,0.0,1,12,B,C,A,...,1.0,1.0,0.0,1.0,1,88.8,1.0,1.0,1,1
1514424,61470349,56425953,53,6327738,0.0,1,12,B,B,A,...,1.0,1.0,0.0,1.0,1,88.8,1.0,1.0,2,1
1514429,61470349,56414487,53,6327738,0.0,1,12,A,B,A,...,1.0,1.0,0.0,1.0,1,88.8,1.0,1.0,1,1
1514430,61470349,56414488,53,6327738,0.0,1,12,A,B,A,...,1.0,1.0,0.0,1.0,1,88.8,1.0,1.0,2,1
1514431,61470349,56414489,53,6327738,0.0,1,12,A,C,A,...,1.0,1.0,0.0,1.0,1,88.8,1.0,1.0,1,2
1514432,61470349,56414490,53,6327738,0.0,1,12,A,B,A,...,1.0,1.0,0.0,1.0,1,88.8,1.0,1.0,1,2
1514433,61470349,56414493,53,6327738,0.0,1,12,A,C,A,...,1.0,1.0,0.0,1.0,1,88.8,1.0,1.0,1,1
1514434,61470349,56414494,53,6327738,0.0,1,12,A,B,A,...,1.0,1.0,0.0,1.0,1,88.8,1.0,1.0,2,2
1514436,61470349,56415763,53,6327738,0.0,1,12,A,D,A,...,1.0,1.0,0.0,1.0,1,88.8,1.0,1.0,2,1


In [23]:
from collections import Counter

# Limpar linhas com nulos
df_final = df_final.dropna()

print(Counter(df_final['NIVEL_LP']))
print(Counter(df_final['NIVEL_MT']))

Counter({1: 118517, 2: 95228, 3: 22392})
Counter({1: 146159, 2: 81369, 3: 8609})


In [39]:
# ==========================================
# Preparação dos Dados para Treinamento e Teste
# ==========================================

df_treino = df_final.copy()

# Removendo IDs e variáveis alvo do conjunto do treino
cols_id = ['ID_ESCOLA', 'ID_ALUNO', 'ID_MUNICIPIO']
targets = ['NIVEL_LP', 'NIVEL_MT']
X = df_treino.drop(columns=cols_id + targets).copy()
y_lp = df_treino['NIVEL_LP']
y_mt = df_treino['NIVEL_MT']

# Conversão de colunas categóricas para numéricas
cols_nominal = ['TX_RESP_Q01', 'TX_RESP_Q03', 'TX_RESP_Q04']
cols_ordinal = [
    c for c in X.columns
    if c.startswith('TX_')
    and c not in cols_nominal
    and X[c].dtype == 'object'
]

mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'G': 5, 'H': 6, 'I': 7}
X[cols_ordinal] = X[cols_ordinal].replace(mapping)



# Treinamento dos modelos

### Modelo 1: Regressão Logística

##### Construção, treinamento e métricas

In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


# Realizando one-hot enconding
X = pd.get_dummies(X, columns=cols_nominal, drop_first=True)
X = pd.get_dummies(X, drop_first=True)


modelo_log = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs'
)

pipeline_log = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', modelo_log)
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def rodar_modelo(nome, y):

    print("\n" + "="*60)
    print(nome)
    print("="*60)

    # ==================================================
    # Split 70 / 15 / 15
    # ==================================================

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        stratify=y,
        random_state=42
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=42
    )

    # ==================================================
    # Cross Validation 
    # ==================================================

    cv_results = cross_validate(
        pipeline_log,
        X_train,
        y_train,
        cv=skf,
        scoring=['accuracy', 'f1_macro', 'f1_weighted'],
        n_jobs=-1
    )

    print("\n── 5-Fold CV (TREINO 70%) ──")
    print(f"Accuracy:   {cv_results['test_accuracy'].mean():.4f} (+/- {cv_results['test_accuracy'].std():.4f})")
    print(f"F1 Macro:   {cv_results['test_f1_macro'].mean():.4f} (+/- {cv_results['test_f1_macro'].std():.4f})")
    print(f"F1 Weighted:{cv_results['test_f1_weighted'].mean():.4f} (+/- {cv_results['test_f1_weighted'].std():.4f})")

    # ==================================================
    # Treino Final (70%)
    # ==================================================

    pipeline_log.fit(X_train, y_train)

    print("\n── PARÂMETROS DO MODELO ──")
    print(pipeline_log.named_steps['modelo'].get_params())

    # ==================================================
    # Validação (15%)
    # ==================================================

    y_val_pred = pipeline_log.predict(X_val)

    print("\n── VALIDAÇÃO (15%) ──")
    print("Accuracy:", accuracy_score(y_val, y_val_pred))
    print("F1 Macro:", f1_score(y_val, y_val_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_val, y_val_pred, average='weighted'))
    print(classification_report(y_val, y_val_pred))
    print(confusion_matrix(y_val, y_val_pred))

    # ==================================================
    # Teste final (15%)
    # ==================================================

    y_test_pred = pipeline_log.predict(X_test)

    print("\n── TESTE FINAL (15%) ──")
    print("Accuracy:", accuracy_score(y_test, y_test_pred))
    print("F1 Macro:", f1_score(y_test, y_test_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_test, y_test_pred, average='weighted'))
    print(classification_report(y_test, y_test_pred))
    print(confusion_matrix(y_test, y_test_pred))

# Aplicação do modelo
rodar_modelo("NIVEL_LP", y_lp)
rodar_modelo("NIVEL_MT", y_mt)


NIVEL_LP

── 5-Fold CV (TREINO 70%) ──
Accuracy:   0.4808 (+/- 0.0019)
F1 Macro:   0.4246 (+/- 0.0017)
F1 Weighted:0.4881 (+/- 0.0017)

── PARÂMETROS DO MODELO ──
{'C': 1.0, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 1000, 'n_jobs': None, 'penalty': 'deprecated', 'random_state': None, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}

── VALIDAÇÃO (15%) ──
Accuracy: 0.47672284802800596
F1 Macro: 0.4216576513510712
F1 Weighted: 0.4841235563975591
              precision    recall  f1-score   support

           1       0.65      0.63      0.64     17777
           2       0.46      0.26      0.33     14285
           3       0.19      0.58      0.29      3359

    accuracy                           0.48     35421
   macro avg       0.43      0.49      0.42     35421
weighted avg       0.53      0.48      0.48     35421

[[11167  3734  2876]
 [ 5232  3772  5281]
 [  669   743  1947]]

── TESTE

##### Ajuste de parâmetros

In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


def rodar_modelo(nome, y):

    print("\n" + "="*60)
    print(nome)
    print("="*60)

    # ==================================================
    # Split70 / 15 / 15
    # ==================================================
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        stratify=y,
        random_state=42
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=42
    )

    # ==================================================
    # Ajuste manual de hiperparâmetros
    # ==================================================
    configs = [
        (1, 'lbfgs'),
        (10,  'lbfgs'),
        (100,  'lbfgs'),
        (1000, 'lbfgs'),
        (1.0,  'saga')
    ]

    melhor_modelo = None
    melhor_f1 = -1

    print("\n── AJUSTE MANUAL (VALIDAÇÃO 15%) ──")

    for C, solver in configs:

        model = Pipeline([
            ('scaler', StandardScaler()),
            ('modelo', LogisticRegression(
                C=C,
                solver=solver,
                max_iter=1000,
                class_weight='balanced'
            ))
        ])

        model.fit(X_train, y_train)
        y_val_pred = model.predict(X_val)

        f1 = f1_score(y_val, y_val_pred, average='macro')

        print(f"C={C}, solver={solver} → F1 macro = {f1:.4f}")

        if f1 > melhor_f1:
            melhor_f1 = f1
            melhor_modelo = model

    # ==================================================
    # Teste final (15%)
    # ==================================================
    y_test_pred = melhor_modelo.predict(X_test)

    print("\n── TESTE FINAL (15%) ──")
    print("Accuracy:", accuracy_score(y_test, y_test_pred))
    print("F1 Macro:", f1_score(y_test, y_test_pred, average='macro'))
    print("F1 Weighted:", f1_score(y_test, y_test_pred, average='weighted'))
    print(classification_report(y_test, y_test_pred))
    print(confusion_matrix(y_test, y_test_pred))


# Aplicação do modelo
rodar_modelo("NIVEL_LP", y_lp)
rodar_modelo("NIVEL_MT", y_mt)


NIVEL_LP

── AJUSTE MANUAL (VALIDAÇÃO 15%) ──
C=1, solver=lbfgs → F1 macro = 0.4217
C=10, solver=lbfgs → F1 macro = 0.4217
C=100, solver=lbfgs → F1 macro = 0.4217
C=1000, solver=lbfgs → F1 macro = 0.4217
C=1.0, solver=saga → F1 macro = 0.4217

── TESTE FINAL (15%) ──
Accuracy: 0.4802800598515005
F1 Macro: 0.42694814044934026
F1 Weighted: 0.48836220488120563
              precision    recall  f1-score   support

           1       0.65      0.62      0.64     17778
           2       0.47      0.28      0.35     14284
           3       0.20      0.59      0.30      3359

    accuracy                           0.48     35421
   macro avg       0.44      0.50      0.43     35421
weighted avg       0.54      0.48      0.49     35421

[[11102  3769  2907]
 [ 5219  3933  5132]
 [  653   729  1977]]

NIVEL_MT

── AJUSTE MANUAL (VALIDAÇÃO 15%) ──
C=1, solver=lbfgs → F1 macro = 0.4392
C=10, solver=lbfgs → F1 macro = 0.4391
C=100, solver=lbfgs → F1 macro = 0.4391
C=1000, solver=lbfgs → F1 macr

### Modelo 2: KNN